# Step 1: Install & Import Dependencies

In [9]:
pip install dash plotly pandas

In [10]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

# Step 2: Load & Prepare Data

In [11]:
# Load data
df = pd.read_csv("NIFTY50_all.csv")

# Convert Date
df['Date'] = pd.to_datetime(df['Date'])

# Ensure numeric columns
num_cols = ['Open','High','Low','Close','Volume','Turnover','Deliverable Volume','%Deliverble']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Sort
df = df.sort_values(by=['Symbol','Date'])


In [12]:
def add_volatility(df, window=20):
    df = df.copy()
    df['Daily Return'] = df['Close'].pct_change()
    df['Rolling Volatility'] = df['Daily Return'].rolling(window).std() * (252**0.5)  # annualized
    return df


# Step 3: Create Dash App Layout

In [13]:
app = Dash(__name__)

app.layout = html.Div([
    html.H1("NIFTY50 Stock Dashboard", style={'textAlign':'center'}),

    # Stock selector
    dcc.Dropdown(
        id='stock-selector',
        options=[{'label': sym, 'value': sym} for sym in df['Symbol'].unique()],
        value='RELIANCE',
        multi=False
    ),

    # Date range picker
    dcc.DatePickerRange(
        id='date-range',
        min_date_allowed=df['Date'].min(),
        max_date_allowed=df['Date'].max(),
        start_date=df['Date'].min(),
        end_date=df['Date'].max()
    ),

    # Charts
    dcc.Graph(id='price-chart'),
    dcc.Graph(id='volume-chart'),
    dcc.Graph(id='returns-dist'),
    dcc.Graph(id='volatility-chart')
])


# Step 4: Add Callbacks for Interactivity

In [14]:
@app.callback(
    [Output('price-chart','figure'),
     Output('volume-chart','figure'),
     Output('returns-dist','figure'),
     Output('volatility-chart','figure')],
    [Input('stock-selector','value'),
     Input('date-range','start_date'),
     Input('date-range','end_date')]
)
def update_dashboard(stock, start_date, end_date):
    dff = df[(df['Symbol']==stock) &
             (df['Date']>=start_date) &
             (df['Date']<=end_date)].copy()

    # Add returns + volatility
    dff = add_volatility(dff)

    # Add daily change for volume chart coloring
    dff['Daily Change'] = dff['Close'].diff()
    dff['Volume Color'] = dff['Daily Change'].apply(lambda x: 'green' if x > 0 else ('red' if x < 0 else 'gray'))


    # Price chart (candlestick)
    price_fig = go.Figure(data=[
        go.Candlestick(
            x=dff['Date'],
            open=dff['Open'],
            high=dff['High'],
            low=dff['Low'],
            close=dff['Close'],
            name="Price"
        )
    ])
    price_fig.update_layout(title=f"{stock} Price Movement")

    # Volume chart with color based on daily change
    vol_fig = px.bar(dff, x='Date', y='Volume', title=f"{stock} Volume",
                     color='Volume Color', color_discrete_map={'green':'green', 'red':'red', 'gray':'gray'})

    # Returns distribution
    ret_fig = px.histogram(dff, x='Daily Return', nbins=50,
                           title=f"{stock} Daily Return Distribution")

    # Rolling volatility chart
    volat_fig = px.line(dff, x='Date', y='Rolling Volatility',
                        title=f"{stock} Rolling Volatility (20D)")

    return price_fig, vol_fig, ret_fig, volat_fig

# Step 5: Run App

In [15]:
if __name__ == '__main__':
    app.run(debug=True)


<IPython.core.display.Javascript object>